# Flood Susceptibility Classification with Logistic Regression

**Portfolio project | Civil & Environmental Engineering | Machine Learning**

## Important data note
This notebook uses a **synthetic teaching dataset** created for learning. It is not observed flood data, and the labels are generated from an illustrative probability rule. Results must not be used for real flood warnings, engineering decisions, or claims about actual catchments.

## Objective
Build a reproducible binary-classification workflow to demonstrate how Logistic Regression can classify records as:
- `0`: No flood
- `1`: Flood

## Questions
1. What features are included in the dataset?
2. How well does the classifier distinguish the two classes on a held-out test set?
3. What do accuracy, precision, recall, F1-score, and the confusion matrix tell us?
4. Why should the results not be interpreted as real-world flood-prediction performance?

## Workflow
Load data → inspect data → define `X` and `y` → split → train a pipeline → predict → evaluate → inspect coefficients → test a new example.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score
)

## 2. Load the dataset

In [ ]:
df = pd.read_csv("flood_susceptibility_synthetic.csv")
df.head()

## 3. Inspect the dataset

In [ ]:
print("Rows and columns:", df.shape)
print("\nColumn types and non-null counts:")
df.info()

In [ ]:
df.describe()

In [ ]:
print("Missing values per column:")
df.isnull().sum()

In [ ]:
print("Class counts:")
df["flood"].value_counts().sort_index()

## 4. Explore the data

The following plots are descriptive views of the **synthetic** data only. Patterns here are not evidence of relationships in real catchments.

In [ ]:
df["flood"].value_counts().sort_index().plot(kind="bar")
plt.xlabel("Flood class (0 = No flood, 1 = Flood)")
plt.ylabel("Number of records")
plt.title("Class distribution in synthetic dataset")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
plt.scatter(df["rainfall_mm"], df["soil_moisture_percent"], c=df["flood"], alpha=0.65)
plt.xlabel("Rainfall (mm)")
plt.ylabel("Soil moisture (%)")
plt.title("Synthetic records: rainfall vs soil moisture")
plt.tight_layout()
plt.show()

## 5. Define features (`X`) and target (`y`)

- **Features (`X`)** are the input variables used by the model.
- **Target (`y`)** is the class we want to predict.
- `flood` is the target: `0` means no flood and `1` means flood.

In [ ]:
feature_columns = [
    "rainfall_mm",
    "temperature_c",
    "soil_moisture_percent",
    "slope_percent"
]

X = df[feature_columns]
y = df["flood"]

print("X shape:", X.shape)
print("y shape:", y.shape)

## 6. Split into training and testing sets

The test set is held out for evaluation. `stratify=y` helps maintain the class proportions in each split.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

## 7. Build and train the Logistic Regression model

`StandardScaler` standardizes the input features. A pipeline keeps scaling and model fitting together and helps avoid fitting the scaler on the test set.

In [ ]:
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

model.fit(X_train, y_train)

## 8. Predict the test set

In [ ]:
y_pred = model.predict(X_test)
y_probability = model.predict_proba(X_test)[:, 1]

comparison = pd.DataFrame({
    "Actual_flood": y_test.to_numpy(),
    "Predicted_flood": y_pred,
    "Estimated_flood_probability": y_probability
})

comparison.head(10)

## 9. Evaluate the classifier

- **Accuracy:** share of all test cases classified correctly.
- **Precision for class 1:** among cases predicted as flood, the share labelled flood in this synthetic dataset.
- **Recall for class 1:** among actual class-1 test cases, the share detected by the model.
- **F1-score:** harmonic mean of precision and recall.
- **Confusion matrix:** counts correct and incorrect predictions by class.
- **ROC-AUC:** measures ranking discrimination from predicted probabilities; it is most useful when both classes appear in the test set.

These metrics describe performance on this generated dataset only.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {accuracy:.3f}")
print("\nClassification report:")
print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=["No flood", "Flood"],
    zero_division=0
))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No flood", "Flood"]
)
disp.plot()
plt.title("Confusion matrix — synthetic test set")
plt.tight_layout()
plt.show()

In [ ]:
if len(np.unique(y_test)) == 2:
    print(f"ROC-AUC: {roc_auc_score(y_test, y_probability):.3f}")
else:
    print("ROC-AUC not calculated because the test set contains only one class.")

## 10. Inspect model coefficients

Because this pipeline standardizes features, the coefficients describe the effect of a **one-standard-deviation change** in each input on the model's log-odds, holding other inputs fixed. A positive coefficient moves the model toward class 1; a negative coefficient moves it toward class 0. These are model associations in synthetic data, not causal effects.

In [ ]:
logistic_model = model.named_steps["logisticregression"]

coefficients = pd.DataFrame({
    "Feature": feature_columns,
    "Coefficient": logistic_model.coef_[0]
}).sort_values("Coefficient", ascending=False)

coefficients

In [ ]:
coefficients.sort_values("Coefficient").plot(
    x="Feature", y="Coefficient", kind="barh", legend=False
)
plt.xlabel("Coefficient (standardized features)")
plt.title("Logistic Regression coefficients")
plt.tight_layout()
plt.show()

## 11. Predict a new illustrative case

This demonstrates the API only. The probability is not a reliable real-world flood risk estimate.

In [ ]:
new_case = pd.DataFrame({
    "rainfall_mm": [280],
    "temperature_c": [26],
    "soil_moisture_percent": [82],
    "slope_percent": [9]
})

predicted_class = model.predict(new_case)[0]
flood_probability = model.predict_proba(new_case)[0, 1]

print("Predicted class:", int(predicted_class))
print("Illustrative estimated flood probability:", round(float(flood_probability), 3))

## 12. Key findings and limitations

Complete this section after running the notebook:
- Record the test-set accuracy, precision, recall, F1-score, and ROC-AUC if available.
- Discuss whether false negatives (missed class-1 cases) occurred.
- Do not claim the model predicts real floods accurately: the dataset is synthetic and the labels are generated, not observed.
- Synthetic relationships can make evaluation look better or worse than performance on real observations.

## 13. What would be needed for a real research project?

1. A documented dataset with observed flood/non-flood labels and clear spatial and temporal coverage.
2. Rainfall, antecedent soil moisture, terrain, land cover, drainage, and other justified predictors.
3. A split that respects time and/or catchment boundaries to reduce leakage.
4. Comparison against simple baselines and other classifiers.
5. Class-imbalance checks, calibration, uncertainty assessment, and independent validation.
6. Clear reporting of data sources, preprocessing, limitations, and reproducibility.

## Conclusion

This project demonstrates a complete scikit-learn binary-classification workflow with Logistic Regression, including data inspection, train/test splitting, probability estimation, evaluation metrics, and coefficient interpretation. It is a learning portfolio project—not an operational flood forecasting model.